In [ ]:
# make autoreload cell
%load_ext autoreload
%autoreload 2



In [ ]:
import os
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
datapath = Path("/Users/rikhoekstra/develop/republic_delegates_data/1705_1795/consolidated/")
attendances = pd.read_parquet(datapath / "attendance_long_1705_1795_v1.0-rc1.parquet")
delegates = pd.read_parquet(datapath / "delegates_1705_1795_v1.0-rc2.parquet")

In [ ]:
attendances

,Kolom1,offset,end,class,oud id,delegate_id,delegate_name,delegate_score,d,m,...,maxjaar,pattern,heerlijkheid,abbrd_status,missing_fields,birth_year,death_year,name_mismatch,pattern_is_valid,age_at_event
0,427232,353,360,delegate,None,13231,"Jordens, Gerrit Dqavid",0.0,17,2,...,1795.0,Jordens;Martis den 17 Maart 1795 Het eerste ja...,None,❌,2.0,NaN,NaN,False,True,NaN
1,427245,276,283,delegate,None,236,"Jordens, Gerrit Dqavid",0.0,18,2,...,NaN,Jordens;Martis den 17 Maart 1795 Het eerste ja...,None,❌,2.0,NaN,NaN,False,True,NaN
2,427257,225,232,delegate,None,236,"Jordens, Gerrit Dqavid",0.0,19,2,...,NaN,Jordens;Martis den 17 Maart 1795 Het eerste ja...,None,❌,2.0,NaN,NaN,False,True,NaN
3,427271,261,268,delegate,None,236,"Jordens, Gerrit Dqavid",0.0,20,2,...,NaN,Jordens;Martis den 17 Maart 1795 Het eerste ja...,None,❌,2.0,NaN,NaN,False,True,NaN
4,427282,278,285,delegate,None,236,"Jordens, Gerrit Dqavid",0.0,23,2,...,NaN,Jordens;Martis den 17 Maart 1795 Het eerste ja...,None,❌,2.0,NaN,NaN,False,True,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
420344,287913,237,240,delegate,None,21971,van Canter,0.0,6,4,...,1760.0,van Canter;Van Canter;Can;Chan,None,✅,2.0,NaN,NaN,False,True,NaN
420345,359725,505,508,delegate,None,21971,van Canter,0.0,19,1,...,1760.0,van Canter;Van Canter;Can;Chan,None,✅,2.0,NaN,NaN,False,True,NaN
420346,359799,387,390,delegate,None,21971,van Canter,0.0,23,1,...,1760.0,van Canter;Van Canter;Can;Chan,None,✅,2.0,NaN,NaN,False,True,NaN
420347,394980,434,437,delegate,None,21971,van Canter,0.0,13,3,...,1760.0,van Canter;Van Canter;Can;Chan,None,✅,2.0,NaN,NaN,False,True,NaN


In [ ]:
attendances.columns

Index(['Kolom1', 'offset', 'end', 'class', 'oud id', 'delegate_id',
       'delegate_name', 'delegate_score', 'd', 'm', 'j', 'lowerpattern',
       'name', 'namens', 'status', 'opmerkingen', 'niettevinden', 'correcties',
       'cons_id_str', 'fullname', 'voornaam', 'tussenvoegsel', 'geslachtsnaam',
       'geboortejaar', 'overlijdensjaar', 'provincie', 'resolutie_refs',
       'minjaar', 'maxjaar', 'pattern', 'heerlijkheid', 'abbrd_status',
       'missing_fields', 'birth_year', 'death_year', 'name_mismatch',
       'pattern_is_valid', 'age_at_event'],
      dtype='object')

In [ ]:
delegate_patterns = attendances.groupby('delegate_id').agg({'lowerpattern': lambda x: ';'.join(x.unique())}).reset_index()

In [ ]:
# we are going to make a new rc, consisting of the following fields
fields = ['offset', 'end', 'class', 'oud id', 'delegate_id',
       'delegate_name','lowerpattern', 'pattern']

rc3 = attendances[fields].copy()
rc3.to_parquet(datapath / "attendance_long_1705_1795_v1.0-rc3.parquet", index=False)


In [ ]:
# now for the delegates, we want to add the patterns as well, so we can use that for matching as well. We will do this by merging the attendances with the delegates on the delegate_id, and then keeping only the relevant fields.
delegates_with_patterns = delegates.merge(delegate_patterns.drop_duplicates(), left_on='delegate_id', right_on='delegate_id', how='left')

In [ ]:
delegates_with_patterns.to_parquet(datapath / "delegates_with_patterns_1705_1795_v1.0_rc3.parquet", index=False)